# Problem 3: Script

In Problem #2 we implemented code that generates a PNG plot image file of hourly Ticker data for past 5 days for the Close Price of Stocks META, AAPL, AMZN, NFLX, GOOG   
We now need migrate the notebook's code cells into a standalone python script ... The script should have functions   
* readCSV(csv_dir)  
* plot_data(df_stocks)  
  
We should also look at generating other python scripts that encapsulate the management of :
* Logging  
* Application Configuration   
  
Furthermore, we should also look at using OOP (Object Orientated Programming) for cleaner encapsulation, re-useability, and split responsibility of the different classes/scripts.   
   
***  

# Application Configuration:
With project application moving closer to full automation, I decided to utilise Python's Config Parser library.  
There is a lot of data hardcoded into script  
1. File path Locations (data/archive/logging/staging)  
1. Stock Ticker details (what stocks, what intervals/periods)  
1. Logging configuration settings (Logging formats, Logging levels, etc.)  
  
This data needs to be better managed by storing in a config files ...   
https://www.youtube.com/watch?v=mukJVR-GRcQ  
https://www.reddit.com/r/Python/comments/w1utza/how_are_you_all_handling_config_files/  

https://docs.python.org/3/library/configparser.html   
https://realpython.com/ref/stdlib/configparser/   
https://towardsdev.com/configuring-like-a-pro-a-comprehensive-guide-to-pythons-configparser-26c49b898629   

This built-in library is used for reading, writing, and managing text-based INI-style configuration files.  
The ini file defined in this project stores :


The ini used in this project is stored here : [app_setting.ini](app_settings.ini)  

The code for managing this ini configuration file utilizes python's configparser from the standard library.  


In [1]:
import configparser

config = configparser.ConfigParser()
config.read("app_settings.ini")

plot_dir = config.get('Folders', 'plot_dir')
dest_dir = config.get('Folders', 'dest_dir')
staging_dir = config.get('Folders', 'staging_dir')
archive_dir = config.get('Folders', 'archive_dir')

print(f"plot_dir : {plot_dir}")
print(f"dest_dir : {dest_dir}")
print(f"staging_dir : {staging_dir}")
print(f"archive_dir : {archive_dir}")


plot_dir : ./data/plots/
dest_dir : ./data/
staging_dir : ./data/staging/
archive_dir : ./data/archive/


This "config" code is encapsulated into a separate file using Object-Oriented Programming (OOP)  
This removes most of config management code from the main script and allows easier maintenance.   
This "utility" class is stored here : [app_settings.py](app_settings.py)

In [2]:
import configparser
class app_config:
    def __init__(self):
        try:
            self.config_file = 'app_settings.ini'     #Hardcoded ... should not change!
            # Create a ConfigParser object
            self.config = configparser.ConfigParser()
            # Read the configuration file
            self.config.read(self.config_file)
        except FileNotFoundError as e:
            print(f"Error: Cannot find config file {self.config_file} :{e}")
        except configparser.ParsingError as e:
            print(f"Config file {self.config_file} is badly formatted: {e}")
        except Exception as e:
            print(f"An unexpected error occurred during loading config file {self.config_file}:", e)

    def getLoggingSettings(self):
        # Logging values from the configuration file
        try:
            logging_active_ini = self.config.getboolean('Logging', 'log_active')
            logging_active = '1' if (logging_active_ini) else '0'
            logging_filename = self.config.get('Logging', 'log_filename')
            logging_level = self.config.get('Logging', 'log_level')
            logging_format = self.config.get('Logging', 'log_format')
            logging_settings = {
                'active': logging_active,
                'filename': logging_filename,
                'level': logging_level,
                'format': logging_format
                }
            return logging_settings
        except configparser.NoSectionError as e:
            print(f"Missing section in config: {e}")
        except configparser.NoOptionError as e:
            print(f"Missing option in config: {e}")
        except Exception as e:
            print(f"An unexpected error occurred during loading config Folder section in file: {self.config_file}: {e}")

    def getFolderSettings(self):
        # Folder path values from the configuration file
        try:
            csv_dir = self.config.get('Folders', 'csv_dir')
            plot_dir = self.config.get('Folders', 'plot_dir')
            dest_dir = self.config.get('Folders', 'dest_dir')
            staging_dir = self.config.get('Folders', 'staging_dir')
            archive_dir = self.config.get('Folders', 'archive_dir')
            filename_format = self.config.get('Folders', 'filename_format')

            folder_settings = {
                'csv_dir' : csv_dir,
                'plot_dir' : plot_dir,
                'dest_dir' : dest_dir,
                'staging_dir' : staging_dir,
                'archive_dir' : archive_dir,
                'filename_format' : filename_format
                }
            return folder_settings
        except configparser.NoSectionError as e:
            print(f"Missing section in config: {e}")
        except configparser.NoOptionError as e:
            print(f"Missing option in config: {e}")
        except Exception as e:
            print(f"An unexpected error occurred during loading config Logging section in file: {self.config_file}: {e}")

    def getStocksSettings(self):
        # Stock Ticker values from the configuration file
        try:
            stock_tickers = self.config.get('Stocks', 'tickers')
            stock_period = self.config.get('Stocks', 'period')
            stock_interval = self.config.get('Stocks', 'interval')
            stock_settings = {
                'tickers' : stock_tickers,
                'period' : stock_period,
                'interval' : stock_interval
                }
            return stock_settings
        except configparser.NoSectionError as e:
            print(f"Missing section in config: {e}")
        except configparser.NoOptionError as e:
            print(f"Missing option in config: {e}")
        except Exception as e:
            print(f"An unexpected error occurred during loading config Logging section in file: {self.config_file}: {e}")


    def getAllSettings(self):
        # Return a distionary of dictionaries (with the retrieved values)
        logging_settings = self.getLoggingSettings()
        folder_settings = self.getFolderSettings()
        stock_settings = self.getStocksSettings()
        config_settings = {'logging':logging_settings, 'folder':folder_settings, 'stock':stock_settings}
        return config_settings



# Main Script

The main Python script called "faang.py" contains the core functions for :
* Downloading yFinance data  
    Function **getFAANGData()** reads config setting for what stock data to download, the interval, and the period of time.  
    The yFinace library is used to extract data (based on configuration) and return a dataframe  
* Archiving files  
    The **archiveData()** function moves any csv files in 'data' folder into archive folder (before staging folder file is moved to data folder)   
* Saving data to a comma-separated file with a timestamped file name  
* Reading latest csv datafile  
* Generating a lineplot based on csv data  
* Saving plot as a PNG file with a timestamped file name  



This script is located here ... [faang.py](faang.py)
